# Football Era Lab: the same engine as the browser

This notebook runs the game's own JavaScript engine through Node, so every number here is the number the browser would show. Python only reads the bundled data, starts Node and displays what comes back. Run all cells from the repository root.

You can change the settings below to try any manager, formation and fifteen players, or let the notebook make a seeded draft for you. The browser is where you make every pick and placement yourself.

In [1]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess

import pandas as pd
from IPython.display import display

# R: repository root; G: bundled game data; NODE: executable running the shared engine.
R = Path.cwd()
if not (R / 'data' / 'game.json').exists() and (R.parent / 'data' / 'game.json').exists():
    R = R.parent
if not (R / 'data' / 'game.json').exists():
    raise FileNotFoundError('Run this notebook in the Football Era Lab repository with data/game.json.')
NODE = shutil.which('node')  # Or set an absolute node.exe path here.
if not NODE:
    raise FileNotFoundError('Install Node 20 or later, or set NODE to its executable path.')
G = json.loads((R / 'data' / 'game.json').read_text(encoding='utf-8'))
print('Node:', subprocess.check_output([NODE, '--version'], text=True).strip())
print('Data SHA-256:', hashlib.sha256((R / 'data' / 'game.json').read_bytes()).hexdigest())
print('Data build:', G.get('meta', {}).get('v'))


Node: v25.8.2
Data SHA-256: 1b6be710feac925f55bab19bb3a4cabdf8cc5bf82cbfe1b9dc7d225a77018537
Data build: 2026-10-07


## Where every rating comes from

Each card is one person at one club in one decade. Its rating, its natural positions, its rating in every pitch slot and its six face stats all come from the same source, shown by a code:

| Code | Source |
| --- | --- |
| `f` | EA FIFA / FC, an edition inside the player's stint at this club (FIFA 07 to FC 26) |
| `c` | Championship Manager 01/02 database at this club, put on EA's scale |
| `i` | EA Icon or Hero card |
| `n` | EA, a season within two years of the stint, adjusted for age |
| `m` | Championship Manager, a season within two years of the stint, adjusted for age |
| `e` | Estimated by a model fitted on the EA and CM ratings, where no game database rates the player |

The table counts cards by decade and source. The 1950s to 1970s are still mostly estimates: no openly published game database covers them, and the Championship Manager retro seasons for those years are added as they arrive.

In [2]:
rows = [dict(decade=int(k.split(':')[1]), source=c['s']) for k, Q in G['cards'].items() for c in Q]
display(pd.crosstab(pd.DataFrame(rows).decade, pd.DataFrame(rows).source, margins=True))
catalogue = pd.DataFrame([
    {'manager': m['nm'], 'attack_grade': m['ga'], 'defence_grade': m['gd'], 'formations': ', '.join(m['f'])}
    for m in G['managers']
])
print('Managers:', len(catalogue), '· formations:', ', '.join(G['formations']))
print('Unique people:', len(G['people']), '· club-decades:', len(G['combos']), '· cards:', len(rows))

source,c,e,f,i,m,n,All
decade,,,,,,,
1950,0,1585,0,2,0,0,1587
1960,0,2238,0,7,0,0,2245
1970,0,2710,0,11,0,0,2721
1980,1089,1867,0,30,562,0,3548
1990,2578,396,0,68,1842,0,4884
2000,1273,1158,1126,73,898,1668,6196
2010,0,399,3686,10,46,1556,5697
2020,203,30,1369,0,23,270,1895
All,5143,10383,6181,201,3371,3494,28773


Managers: 96 · formations: 2-3-5, WM, 4-2-4, Catenaccio, 4-4-2, 4-4-2 diamond, 4-3-3, 4-3-3 false nine, 4-2-3-1, 4-3-2-1, 4-3-1-2, 4-1-4-1, 3-5-2, 3-4-3, 3-4-3 diamond, 3-4-2-1, 5-3-2
Unique people: 16470 · club-decades: 505 · cards: 28773


## One player, every slot

A player's strengths and weaknesses decide where he is worth playing. Where a game database rates him, the engine reads his rating in each slot from his card: EA's own per-position ratings, or his Championship Manager attributes put on EA's scale. Playing him in a slot rated below his overall costs exactly that gap. Change `who` to look at someone else.

In [3]:
SL = ['GK', 'LB', 'CB', 'RB', 'LWB', 'RWB', 'CDM', 'CM', 'CAM', 'LM', 'RM', 'LW', 'RW', 'CF', 'ST']
who, at = 'Diego Maradona', 'Napoli'
hits = [(k, c) for k, Q in G['cards'].items() for c in Q
        if G['people'][c['p']]['nm'] == who and at.casefold() in G['clubs'][k.split(':')[0]]['nm'].casefold()]
for k, c in hits:
    print(f"{who}, {at} {k.split(':')[1]}s: rating {c['r']} (source {c['s']}), natural slots {', '.join(c['pos'])}")
    if 'sr' in c:
        display(pd.DataFrame([c['sr']], columns=SL, index=['rating in slot']))
    if 'f6' in c:
        names = ['DIV', 'HAN', 'KIC', 'REF', 'SPD', 'POS'] if c['pos'][0] == 'GK' else ['PAC', 'SHO', 'PAS', 'DRI', 'DEF', 'PHY']
        display(pd.DataFrame([c['f6']], columns=names, index=['face stats']))
    print(f"League apps {c['n']}, goals {c['g']}", '· real stats:', c.get('st', 'none recorded for this card'))

Diego Maradona, Napoli 1980s: rating 91.2 (source c), natural slots CAM, CF, CM


,GK,LB,CB,RB,LWB,RWB,CDM,CM,CAM,LM,RM,LW,RW,CF,ST
rating in slot,14,70,66,71,74,74,75,90,91,89,89,89,89,90,88


,PAC,SHO,PAS,DRI,DEF,PHY
face stats,75,93,96,95,60,83


League apps 161, goals 69 · real stats: none recorded for this card
Diego Maradona, Napoli 1990s: rating 91.2 (source m), natural slots CAM, CF, CM


,GK,LB,CB,RB,LWB,RWB,CDM,CM,CAM,LM,RM,LW,RW,CF,ST
rating in slot,14,70,66,71,74,74,75,90,91,89,89,89,89,90,88


,PAC,SHO,PAS,DRI,DEF,PHY
face stats,75,93,96,95,60,83


League apps 27, goals 12 · real stats: none recorded for this card


## Configure the experiment

Leave both lists empty for a seeded notebook draft. `person_ids` takes fifteen distinct Wikidata person IDs; with IDs alone, the bridge prefers a card from the simulation decade, then the highest rating. `source_cards` keeps exact club-decade cards, for example `{'k': 'Q2641:1980', 'p': 'Q17515'}` for Maradona's Napoli card.

`placement='best'` lets the engine assign the eleven starters; `placement='ordered'` keeps your order: eleven formation slots, then four substitutes.

In [4]:
settings = {
    'seed': 20261006,
    'decade': 1990,
    'manager': None,  # A manager name from the catalogue, or None for a seeded choice.
    'formation': None,  # A formation name, or None to use that manager option.
    'person_ids': [],
    'source_cards': [],
    'placement': 'best',
}

def engine(settings):
    """Run the repository JavaScript bridge and return its JSON response."""
    p = subprocess.run([NODE, str(R / 'notebooks' / 'engine_bridge.mjs')],
                       input=json.dumps(settings), capture_output=True, text=True,
                       encoding='utf-8', cwd=R, timeout=120, check=False)
    if p.returncode:
        raise RuntimeError(p.stderr.strip() or f'Node exited with code {p.returncode}')
    return json.loads(p.stdout)

result = engine(settings)
print(result['mode'])
print('Manager:', result['manager']['nm'], 'Formation:', result['formation'])
print('Match parameters:')
print(json.dumps(result['parameters'], indent=2))
display(pd.DataFrame(result['squad']))
print('Source counts:', result['source_counts'])
print('Exact source cards for replay:')
print(json.dumps(result['source_cards']))


Notebook-only seeded draft demonstration: automatic picks, with editable placement.
Manager: Erik ten Hag Formation: 4-3-3
Match parameters:
{
  "be": 0.4115033527449246,
  "ka": 0.5566086036333626,
  "h": 0.2620934744673785,
  "pa": 0.05,
  "fat": 0.04,
  "m1": 60,
  "m2": 75,
  "ns": 3,
  "b": {
    "1950": 1.45864386651915,
    "1960": 1.2789984962526806,
    "1970": 1.185731721121677,
    "1980": 1.176241415929074,
    "1990": 1.1535407576488488,
    "2000": 1.1487314074297343,
    "2010": 1.2313444534429265,
    "2020": 1.2313444534429265
  },
  "src": "Fitted on 2014-19 real club-season results; held out 2020-23. Neutral 2020s scoring level is frozen from training; historical decade levels are measured ratios."
}


,place,id,player,club,card_decade,positions,base_rating,source,tags
0,1: GK,Q131261,Oliver Kahn,Bayern Munich,2000,GK,90.2,c,"{""rock"":1,""tl"":2,""bg"":1}"
1,2: LB,Q483577,Rivaldo,AC Milan,2000,"LW, LM, CAM, CF",90.0,i,"{""tal"":1,""tl"":2,""bg"":1}"
2,3: CB,Q312626,Marco Tardelli,Juventus,1980,CM,87.3,e,"{""bg"":1}"
3,4: CB,Q483629,Marco van Basten,AC Milan,1990,"CF, ST",92.0,c,"{""tal"":1,""tl"":1,""poa"":2,""bg"":3}"
4,5: RB,Q160106,Sebastian Deisler,Bayern Munich,2000,"CM, CAM, RM",87.2,m,{}
5,6: CDM,Q68039,Zbigniew Boniek,Juventus,1980,"CM, ST, CF",87.6,e,"{""bg"":1}"
6,7: CM,Q4261,Michel Platini,Juventus,1980,CM,89.3,e,"{""mae"":1,""tl"":1,""bg"":1}"
7,8: CM,Q969725,Harry Kane,Bayern Munich,2020,ST,90.0,f,"{""tal"":1,""poa"":1}"
8,9: LW,Q67964,Roberto Baggio,AC Milan,1990,"CAM, CF, ST",92.0,c,"{""tal"":1,""tl"":1}"
9,10: ST,Q529207,Ronaldo,AC Milan,2000,"ST, CF",92.8,i,"{""tal"":1,""tl"":1}"


Source counts: {'c': 4, 'f': 5, 'i': 2, 'e': 3, 'm': 1}
Exact source cards for replay:
[{"k": "Q15789:2000", "p": "Q131261"}, {"k": "Q1543:2000", "p": "Q483577"}, {"k": "Q1422:1980", "p": "Q312626"}, {"k": "Q1543:1990", "p": "Q483629"}, {"k": "Q15789:2000", "p": "Q160106"}, {"k": "Q1422:1980", "p": "Q68039"}, {"k": "Q1422:1980", "p": "Q4261"}, {"k": "Q15789:2020", "p": "Q969725"}, {"k": "Q1543:1990", "p": "Q67964"}, {"k": "Q1543:2000", "p": "Q529207"}, {"k": "Q15789:2000", "p": "Q1918"}, {"k": "Q15789:2020", "p": "Q107365"}, {"k": "Q15789:2020", "p": "Q151269"}, {"k": "Q1543:2000", "p": "Q62166"}, {"k": "Q1543:1990", "p": "Q522040"}]


## See fit, era and rating changes

The starter table is the engine's own rating breakdown. Position loss comes from the player's slot ratings where his card has them, otherwise from how far the slot is from his natural position. Bench players have no position loss. Links, Timeless and manager grades are game rules.

In [5]:
display(pd.DataFrame(result['starters']))
display(pd.DataFrame(result['bench']))
display(pd.DataFrame([result['lines']]))


,slot,id,player,base_rating,source,card_decade,fit_label,position_loss,era_multiplier,chemistry_points,adjusted_rating
0,GK,Q131261,Oliver Kahn,90.2,c,2000,Natural,0.002217,0.99250,0,89.32500
1,LB,Q483577,Rivaldo,90.0,i,2000,Out of position -22%,0.220000,0.99250,0,69.67350
2,CB,Q312626,Marco Tardelli,87.3,e,1980,Out of position -22%,0.220000,0.97000,2,68.05118
3,CB,Q483629,Marco van Basten,92.0,c,1990,Major penalty -27%,0.271739,1.00000,0,67.00000
4,RB,Q160106,Sebastian Deisler,87.2,m,2000,Adapted -6%,0.059633,0.98500,0,80.77000
5,CDM,Q68039,Zbigniew Boniek,87.6,e,1980,Adapted -10%,0.100000,0.97000,5,81.47480
6,CM,Q4261,Michel Platini,89.3,e,1980,Natural,0.000000,0.99250,5,93.63025
7,CM,Q969725,Harry Kane,90.0,f,2020,Out of position -8%,0.077778,0.95500,0,79.26500
8,LW,Q67964,Roberto Baggio,92.0,c,1990,Adapted -2%,0.021739,1.00000,0,90.00000
9,ST,Q529207,Ronaldo,92.8,i,2000,Natural,0.000000,0.99625,0,92.45200


,id,player,base_rating,source,card_decade,position_loss,era_multiplier,adjusted_rating
0,Q107365,Manuel Neuer,89.7,f,2020,0,0.9550,85.66350
1,Q151269,Robert Lewandowski,91.5,f,2020,0,0.9775,89.44125
2,Q62166,Christian Abbiati,80.5,f,2000,0,0.9850,79.29250
3,Q522040,Sebastiano Rossi,78.2,c,1990,0,1.0000,78.20000


,attack,midfield,outfield_defence,keeper,keeper_weighted_defence,overall,attack_grade,defence_grade,signature_upgrade,knockout_boost
0,93.018625,86.554006,75.745502,89.325,79.819351,82.425644,B,C,False,0.02079


## Play the league and European Cup

Twenty clubs play home and away; sixteen enter the Cup, with two-legged ties, a neutral final, extra time and penalties. Who scores in each simulated goal follows the players' real scoring rates where their cards have them.

In [6]:
season = result['season']
display(pd.DataFrame(season['L']['tab']))
display(pd.DataFrame([
    {k: v for k, v in fixture.items() if k != 'M'} for fixture in season['L']['res']
]))
cup_rows = []
for round_ties in season['K']['rounds']:
    for t in round_ties:
        cup_rows.append({'round': t['k'], 'home': t['an'], 'away': t['bn'],
                         'aggregate': f"{t['agg'][0]}-{t['agg'][1]}", 'winner': t['wn'],
                         'extra_time': t.get('et', False), 'penalties': t.get('pw')})
display(pd.DataFrame(cup_rows))
print('Cup winner:', season['K']['champN'])
display(pd.DataFrame(result['player_stats']))
if 'awards' in result:
    print(json.dumps(result['awards'], indent=2, ensure_ascii=False))


,id,nm,me,P,W,D,L,GF,GA,Pts
0,Q7156:1990,Barcelona · 1990s,False,38,29,4,5,140,37,91
1,Q1543:1990,AC Milan · 1990s,False,38,26,6,6,114,43,84
2,Q1422:1990,Juventus · 1990s,False,38,25,7,6,87,26,82
3,Q15789:1990,Bayern Munich · 1990s,False,38,23,9,6,77,33,78
4,Q8682:1990,Real Madrid · 1990s,False,38,21,9,8,95,56,72
5,Q132885:1990,Marseille · 1990s,False,38,22,6,10,81,47,72
6,Q1457:1990,Sampdoria · 1990s,False,38,23,2,13,77,50,71
7,Q18656:1990,Manchester United · 1990s,False,38,20,6,12,87,49,66
8,Q180305:1990,Monaco · 1990s,False,38,17,10,11,68,57,61
9,Q81888:1990,Ajax · 1990s,False,38,19,4,15,65,68,61


,rd,h,op,gf,ga
0,1,True,Panathinaikos · 1990s,4,0
1,2,NaN,Spartak Moscow · 1990s,1,2
2,3,True,Red Star Belgrade · 1990s,2,0
3,4,NaN,Feyenoord · 1990s,2,1
4,5,True,Dynamo Kyiv · 1990s,4,1
5,6,NaN,Kaiserslautern · 1990s,3,1
6,7,True,PSV Eindhoven · 1990s,2,4
7,8,NaN,Porto · 1990s,1,1
8,9,True,Monaco · 1990s,3,1
9,10,NaN,Ajax · 1990s,2,2


,round,home,away,aggregate,winner,extra_time,penalties
0,Round of 16,Your Era XI,Barcelona · 1990s,2-11,Barcelona · 1990s,False,None
1,Round of 16,Ajax · 1990s,Real Madrid · 1990s,2-2,Real Madrid · 1990s,True,"{'w': 0, 'x': 5, 'y': 3}"
2,Round of 16,Porto · 1990s,Juventus · 1990s,1-3,Juventus · 1990s,False,None
3,Round of 16,Monaco · 1990s,Bayern Munich · 1990s,1-5,Bayern Munich · 1990s,False,None
4,Round of 16,Dynamo Kyiv · 1990s,AC Milan · 1990s,1-9,AC Milan · 1990s,False,None
5,Round of 16,Kaiserslautern · 1990s,Manchester United · 1990s,0-9,Manchester United · 1990s,False,None
6,Round of 16,Borussia Dortmund · 1990s,Marseille · 1990s,3-4,Marseille · 1990s,False,None
7,Round of 16,PSV Eindhoven · 1990s,Sampdoria · 1990s,4-7,Sampdoria · 1990s,False,None
8,Quarter-final,Real Madrid · 1990s,Juventus · 1990s,1-4,Juventus · 1990s,False,None
9,Quarter-final,Sampdoria · 1990s,Bayern Munich · 1990s,3-5,Bayern Munich · 1990s,False,None


Cup winner: Juventus · 1990s


,id,cl,nm,src,k,g,as,cs,ap,competition
0,Q131261,your-club,Oliver Kahn,c,True,0,0,7,37,League
1,Q483577,your-club,Rivaldo,i,False,1,3,0,35,League
2,Q312626,your-club,Marco Tardelli,e,False,1,1,0,35,League
3,Q483629,your-club,Marco van Basten,c,False,2,2,0,38,League
4,Q160106,your-club,Sebastian Deisler,m,False,0,2,0,37,League
...,...,...,...,...,...,...,...,...,...,...
509,Q41244,Q1543:1990,Andriy Shevchenko,i,False,0,0,0,1,European Cup
510,Q313518,Q132885:1990,Alen Bokšić,m,False,0,0,0,1,European Cup
511,Q336916,Q132885:1990,Abedi Pele,i,False,0,0,0,1,European Cup
512,Q44464,Q132885:1990,Rudi Völler,c,False,0,0,0,1,European Cup


{
  "league": {
    "scorer": {
      "id": "Q487155",
      "cl": "Q8682:1990",
      "nm": "Hugo Sánchez",
      "src": "i",
      "k": false,
      "g": 37,
      "as": 2,
      "cs": 0,
      "ap": 36
    },
    "assists": {
      "id": "Q189892",
      "cl": "Q7156:1990",
      "nm": "Gheorghe Hagi",
      "src": "c",
      "k": false,
      "g": 11,
      "as": 21,
      "cs": 0,
      "ap": 37
    },
    "keeper": {
      "id": "Q482955",
      "cl": "Q1422:1990",
      "nm": "Edwin van der Sar",
      "src": "i",
      "k": true,
      "g": 0,
      "as": 0,
      "cs": 22,
      "ap": 36
    },
    "player": {
      "id": "Q483577",
      "cl": "Q7156:1990",
      "nm": "Rivaldo",
      "src": "c",
      "k": false,
      "g": 31,
      "as": 12,
      "cs": 0,
      "ap": 36
    }
  },
  "cup": {
    "scorer": {
      "id": "Q178649",
      "cl": "Q7156:1990",
      "nm": "Romário",
      "src": "c",
      "k": false,
      "g": 7,
      "as": 1,
      "cs": 0,
      "ap": 7


## Replay and keep a result

The same settings, data and engine always give the same season. This cell checks that by running everything twice. Set `export_path` to a folder you keep if you want the full result as JSON; by default nothing is written.

In [7]:
again = engine(settings)
assert result == again, 'The same settings produced a different bridge response.'
print('Deterministic replay matched for these settings and this data bundle.')

export_path = None  # Set a persistent Path to write the full result as JSON.
if export_path is not None:
    p = Path(export_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Wrote:', p.resolve())


Deterministic replay matched for these settings and this data bundle.


## What these numbers can and cannot tell you

Squads come from dated Wikidata club records, so a decade squad can hold players who never shared a season, and apps and goals are spread across the years of each stint. EA and Championship Manager ratings are each game's judgement of a player, put on one scale; the estimates for earlier decades extrapolate a model fitted on 1989-2025 cards. The match model was fitted on real club results from 2014-19 and checked on 2020-23. Era losses, links, tags and manager grades are game rules, not measurements, and every competition uses one modern format.